# Anderson Impurity Model: star and chain geometries

Walkthrough of `AndersonImpurity` (Task 21) -- one impurity orbital coupled to a bath of non-interacting levels, in two independently-specified geometries: **star** (impurity coupled directly to every bath mode) and **chain** (impurity coupled only to the first of a nearest-neighbor-hopping chain). There is no conversion between them here (the star-to-chain Lanczos mapping is out of scope) -- both are built directly from their own parameters, exactly like `FermiHubbard1D` is built directly, not converted from some other representation.

Parameters below are small synthetic numbers for demonstration -- not the material-derived values from the source document (which are confidential and stay out of the repository).

In [1]:
import numpy as np
from scipy.linalg import expm

from qaravan.applications import ground_state
from qaravan.core.fermions import FermionicEncoding, bath_interleaved_mode_order, majorana_bilinear
from qaravan.core.hamiltonians import AndersonImpurity

np.set_printoptions(precision=4, suppress=True)

## 1. Star geometry

$H = \varepsilon_d\sum_\sigma d_\sigma^\dagger d_\sigma + U n_{d\uparrow}n_{d\downarrow} + \sum_{k,\sigma}\varepsilon_k c_{k\sigma}^\dagger c_{k\sigma} + \sum_{k,\sigma}(V_k c_{k\sigma}^\dagger d_\sigma + \text{h.c.})$ -- the impurity couples directly to every bath mode, bath modes never couple to each other. `AndersonImpurity` is bare (a `FermionicSum`) until `.encode(...)` is called, exactly like `FermiHubbard1D`.

In [2]:
eps_d, U = -0.6, 1.6
eps_bath = [0.9, -0.4, 0.2]
V = [0.5, 0.35, 0.2]

star = AndersonImpurity.star(eps_d=eps_d, U=U, eps_bath=eps_bath, V=V)
print(type(star).__name__)
print(f"topology={star.topology!r}  num_bath={star.num_bath}  num_modes={star.num_fermionic_modes}")
star

AndersonImpurity
topology='star'  num_bath=3  num_modes=8


0.5 * c†_0 c_1 + 0.5 * c†_1 c_0 + 0.5 * c†_4 c_5 + 0.5 * c†_5 c_4 + 0.35 * c†_0 c_2 + 0.35 * c†_2 c_0 + 0.35 * c†_4 c_6 + 0.35 * c†_6 c_4 + 0.2 * c†_0 c_3 + 0.2 * c†_3 c_0 + 0.2 * c†_4 c_7 + 0.2 * c†_7 c_4 + -0.6 * c†_0 c_0 + -0.6 * c†_4 c_4 + 1.6 * c†_0 c_0 c†_4 c_4 + 0.9 * c†_1 c_1 + 0.9 * c†_5 c_5 + -0.4 * c†_2 c_2 + -0.4 * c†_6 c_6 + 0.2 * c†_3 c_3 + 0.2 * c†_7 c_7

In [3]:
encoding = FermionicEncoding("jordan_wigner", num_modes=star.num_fermionic_modes)  # identity mode_order
star_ham = star.encode(encoding)
H_star = star_ham.to_matrix()
assert np.allclose(H_star, H_star.conj().T)
print("Hermitian: OK, shape", H_star.shape)

Hermitian: OK, shape (256, 256)


## 2. Chain geometry

$H = H_{\rm imp} + \sum_\sigma\left[V(f_{0,\sigma}^\dagger d_\sigma + \text{h.c.}) + \sum_n\left(\tilde\varepsilon_n f_{n,\sigma}^\dagger f_{n,\sigma} + t_n(f_{n,\sigma}^\dagger f_{n+1,\sigma}+\text{h.c.})\right)\right]$ -- the impurity couples only to the first chain site, via a single scalar $V$; chain sites hop only to their nearest neighbor. This is a *different*, independently-specified bath (not derived from the star one above) -- same recipe as `FermiHubbard1D`'s open chain, with the impurity playing the role of "site 0".

In [4]:
eps_chain = [0.5, -0.3, 0.15]
t_chain = [0.8, 0.6]          # len(eps_chain) - 1
V_chain = 0.7

chain = AndersonImpurity.chain(eps_d=eps_d, U=U, eps_chain=eps_chain, t_chain=t_chain, V=V_chain)
print(f"topology={chain.topology!r}  num_bath={chain.num_bath}  num_modes={chain.num_fermionic_modes}")
print(type(chain).__name__)
chain

topology='chain'  num_bath=3  num_modes=8
AndersonImpurity


0.7 * c†_0 c_1 + 0.7 * c†_1 c_0 + 0.6 * c†_2 c_3 + 0.6 * c†_3 c_2 + 0.8 * c†_1 c_2 + 0.8 * c†_2 c_1 + 0.7 * c†_4 c_5 + 0.7 * c†_5 c_4 + 0.6 * c†_6 c_7 + 0.6 * c†_7 c_6 + 0.8 * c†_5 c_6 + 0.8 * c†_6 c_5 + -0.6 * c†_0 c_0 + -0.6 * c†_4 c_4 + 1.6 * c†_0 c_0 c†_4 c_4 + 0.5 * c†_1 c_1 + 0.5 * c†_5 c_5 + -0.3 * c†_2 c_2 + -0.3 * c†_6 c_6 + 0.15 * c†_3 c_3 + 0.15 * c†_7 c_7

In [5]:
chain_encoding = FermionicEncoding("jordan_wigner", num_modes=chain.num_fermionic_modes)
chain_ham = chain.encode(chain_encoding)
H_chain = chain_ham.to_matrix()
assert np.allclose(H_chain, H_chain.conj().T)
print("Hermitian: OK")

Hermitian: OK


## 3. Sanity check: the atomic limit (no bath at all)

With `num_bath=0`, both geometries degenerate to $H_{\rm imp}$ alone -- a clean, closed-form check with no bath machinery involved. The four impurity-only eigenstates ($|0\rangle,\ |\!\uparrow\rangle,\ |\!\downarrow\rangle,\ |\!\uparrow\downarrow\rangle$) have energies $\{0,\ \varepsilon_d,\ \varepsilon_d,\ 2\varepsilon_d+U\}$ exactly.

This can be seen by moving to qubit language where the "computational basis states" ($|00\rangle$, $|01\rangle$, $|10\rangle$, $|11\rangle$) are each an eigenstate of the Hamiltonian since it looks like: $\epsilon_d (n_0+n_1) + Un_0n_1$ where $0,1$ index on the number operators refer to qubit indices. The number operator acting on computational basis gives 0 or 1. 

In [6]:
U = 1.6
eps_d = -0.6
atomic = AndersonImpurity.star(eps_d=eps_d, U=U, eps_bath=[], V=[])
atomic_encoding = FermionicEncoding("jordan_wigner", num_modes=atomic.num_fermionic_modes)

# let's find the 4 eigenstates 
mat = atomic.encode(atomic_encoding).to_matrix()
evals, evecs = np.linalg.eigh(mat)

# turn each evec into Statevector 
from qaravan.backends import Statevector
for i in range(4):
    print(f"eigenvalue {i}: {evals[i]:.4f}")
    print(Statevector(array=evecs[:, i]))

eigenvalue 0: -0.6000
1.0000|01⟩
eigenvalue 1: -0.6000
1.0000|10⟩
eigenvalue 2: 0.0000
1.0000|00⟩
eigenvalue 3: 0.4000
1.0000|11⟩


## 4. A real downfolded material Hamiltonian (5-bath star)

The 5-bath star parameters from Table 1 of the source document ($n_b=5 \Rightarrow 2(n_b+1)=12$ qubits, matching the document's own "12-qubit system" caption -- a nice independent check that the mode-counting convention above is right).

In [7]:
# Table 1, nb=5 row (Ca2CuO2Cl2 downfolded single-band model)
U_mat = 3.232
eps_d_mat = -1.309
eps_bath_mat = [0.735, 1.587, 0.255, 0.065, -0.001]
V_mat = [0.988, 2.432, 1.513, 0.388, 0.155]

star_mat = AndersonImpurity.star(eps_d=eps_d_mat, U=U_mat, eps_bath=eps_bath_mat, V=V_mat)
print(f"num_bath={star_mat.num_bath}, num_modes={star_mat.num_fermionic_modes}")
assert star_mat.num_fermionic_modes == 12

mode_order, annotation = bath_interleaved_mode_order(star_mat.num_bath, annotate=True)
print(annotation)
star_mat_encoding = FermionicEncoding("jordan_wigner", num_modes=star_mat.num_fermionic_modes, mode_order=mode_order)
star_mat_ham = star_mat.encode(star_mat_encoding)
star_mat_ham

num_bath=5, num_modes=12
q0=b1↑ q1=b1↓ q2=b2↑ q3=b2↓ q4=b3↑ q5=b3↓ q6=b4↑ q7=b4↓ q8=b5↑ q9=b5↓ q10=i↑ q11=i↓


((0.494+0j))*XZZZZZZZZZXI + ((0.494+0j))*YZZZZZZZZZYI + ((0.494+0j))*IXZZZZZZZZZX + ((0.494+0j))*IYZZZZZZZZZY + ((1.216+0j))*IIXZZZZZZZXI + ((1.216+0j))*IIYZZZZZZZYI + ((1.216+0j))*IIIXZZZZZZZX + ((1.216+0j))*IIIYZZZZZZZY + ((0.7565+0j))*IIIIXZZZZZXI + ((0.7565+0j))*IIIIYZZZZZYI + ((0.7565+0j))*IIIIIXZZZZZX + ((0.7565+0j))*IIIIIYZZZZZY + ((0.194+0j))*IIIIIIXZZZXI + ((0.194+0j))*IIIIIIYZZZYI + ((0.194+0j))*IIIIIIIXZZZX + ((0.194+0j))*IIIIIIIYZZZY + ((0.0775+0j))*IIIIIIIIXZXI + ((0.0775+0j))*IIIIIIIIYZYI + ((0.0775+0j))*IIIIIIIIIXZX + ((0.0775+0j))*IIIIIIIIIYZY + ((2.14+0j))*IIIIIIIIIIII + ((-0.15350000000000008+0j))*IIIIIIIIIIZI + ((-0.15350000000000008+0j))*IIIIIIIIIIIZ + ((0.808+0j))*IIIIIIIIIIZZ + ((-0.3675+0j))*ZIIIIIIIIIII + ((-0.3675+0j))*IZIIIIIIIIII + ((-0.7935+0j))*IIZIIIIIIIII + ((-0.7935+0j))*IIIZIIIIIIII + ((-0.1275+0j))*IIIIZIIIIIII + ((-0.1275+0j))*IIIIIZIIIIII + ((-0.0325+0j))*IIIIIIZIIIII + ((-0.0325+0j))*IIIIIIIZIIII + ((0.0005+0j))*IIIIIIIIZIII + ((0.0005+0j))*IIIIIIII

### Ground state

12 qubits ⇒ a $4096\times4096$ dense matrix -- exact diagonalization is still trivial at this size.

In [8]:
gs_mat = ground_state(star_mat_ham, method="naive_lanczos")  # only the ground eigenpair, not the whole spectrum
print(type(gs_mat).__name__)

Statevector


In [9]:
E0_mat = gs_mat.expectation(star_mat_ham.as_observable()).real
print(f"ground energy E0 = {E0_mat:.6f}")

# Cross-check against direct diagonalization
E0_direct = np.linalg.eigvalsh(star_mat_ham.to_matrix())[0]
assert np.isclose(E0_mat, E0_direct)

ground energy E0 = -5.678950


### Aside: `ground_state()` was slow here -- `method="naive_lanczos"`

`ground_state()`'s default (`method="naive"`) is `numpy.linalg.eigh` -- it diagonalizes the
*entire* $4096\times4096$ matrix even though only the bottom eigenvector is kept, which is
exactly the O(n³) waste that made this cell slow the first time through. `method="naive_lanczos"`
(used above) calls `scipy.sparse.linalg.eigsh(k=1)` instead -- only the extremal eigenpair, via
Lanczos iteration, O(n²) per iteration rather than diagonalizing everything. Benchmarked here
directly rather than just asserted:

In [10]:
import time

t0 = time.perf_counter()
E0_exact = ground_state(star_mat_ham, method="naive").expectation(star_mat_ham.as_observable()).real
t1 = time.perf_counter()
E0_lanczos = ground_state(star_mat_ham, method="naive_lanczos").expectation(star_mat_ham.as_observable()).real
t2 = time.perf_counter()

print(f"exact:   {t1 - t0:6.2f}s   E0={E0_exact:.8f}")
print(f"lanczos: {t2 - t1:6.2f}s   E0={E0_lanczos:.8f}")
print(f"speedup: {(t1 - t0) / (t2 - t1):.1f}x")
assert np.isclose(E0_exact, E0_lanczos, atol=1e-8)

exact:    38.26s   E0=-5.67894982
lanczos:   4.45s   E0=-5.67894982
speedup: 8.6x


## 5. Cross-check against the equivalent chain

Star and chain are unrelated parameterizations *unless* the chain parameters are actually
derived from the star ones -- which the source document shows how to do (§2.1, eq. 3-7):
Lanczos-tridiagonalize the diagonal bath-energy operator $\hat\varepsilon={\rm diag}(\varepsilon_k)$,
seeded by the normalized hybridization vector $v_k = V_k/V$, $V=\sqrt{\sum_k V_k^2}$. That
produces exactly the $\tilde\varepsilon_n$, $t_n$ of eq. (7)-(8).

This conversion is *not* part of `core/` (per the proposal, out of scope for the library) --
it's a one-off, ad-hoc numpy routine here, used only to generate a genuine cross-check: if it's
implemented correctly, the resulting chain must reproduce the star spectrum *exactly*, for any
$n_b$, not just the trivial $n_b=1$ case the unit tests can check without doing this derivation.

In [11]:
def lanczos_chain_params(eps_bath, V):
    """Notebook-only, ad-hoc star->chain conversion (eq. 3-7): Lanczos-tridiagonalize
    diag(eps_bath) seeded by V/||V||. Returns (eps_chain, t_chain, V_total)."""
    eps_bath = np.asarray(eps_bath, dtype=float)
    V = np.asarray(V, dtype=float)
    num_bath = len(eps_bath)
    V_total = np.sqrt(np.sum(V**2))

    f_curr = V / V_total          # f_0, eq. (3)
    f_prev = np.zeros(num_bath)
    b_prev = 0.0
    eps_chain = np.zeros(num_bath)
    t_chain = np.zeros(num_bath - 1)

    for n in range(num_bath):
        Df = eps_bath * f_curr
        a_n = f_curr @ Df
        eps_chain[n] = a_n
        if n == num_bath - 1:
            break
        r = Df - a_n * f_curr - b_prev * f_prev
        b_n = np.linalg.norm(r)
        t_chain[n] = b_n
        f_prev, f_curr = f_curr, r / b_n
        b_prev = b_n
    return eps_chain, t_chain, V_total


eps_chain_mat, t_chain_mat, V_total_mat = lanczos_chain_params(eps_bath_mat, V_mat)
print("eps_chain:", eps_chain_mat)
print("t_chain:  ", t_chain_mat)
print("V (impurity <-> f_0):", V_total_mat)

eps_chain: [1.1436 0.7299 0.6184 0.1312 0.018 ]
t_chain:   [0.5975 0.2214 0.1914 0.0336]
V (impurity <-> f_0): 3.058513691321325


In [12]:
chain_mat = AndersonImpurity.chain(
    eps_d=eps_d_mat, U=U_mat, eps_chain=eps_chain_mat.tolist(), t_chain=t_chain_mat.tolist(), V=V_total_mat
)
chain_mat_encoding = FermionicEncoding("jordan_wigner", num_modes=chain_mat.num_fermionic_modes)
chain_mat_ham = chain_mat.encode(chain_mat_encoding)
chain_mat_ham # look how much more local the chain Hamiltonian is compared to the star Hamiltonian!

((1.5292568456606626+0j))*YYIIIIIIIIII + ((1.5292568456606626+0j))*XXIIIIIIIIII + ((0.11070343712387927+0j))*IIYYIIIIIIII + ((0.11070343712387927+0j))*IIXXIIIIIIII + ((0.016786997725403114+0j))*IIIIYYIIIIII + ((0.016786997725403114+0j))*IIIIXXIIIIII + ((0.2987659097264251+0j))*IYYIIIIIIIII + ((0.2987659097264251+0j))*IXXIIIIIIIII + ((0.09572277709061083+0j))*IIIYYIIIIIII + ((0.09572277709061083+0j))*IIIXXIIIIIII + ((1.5292568456606626+0j))*IIIIIIYYIIII + ((1.5292568456606626+0j))*IIIIIIXXIIII + ((0.11070343712387927+0j))*IIIIIIIIYYII + ((0.11070343712387927+0j))*IIIIIIIIXXII + ((0.016786997725403114+0j))*IIIIIIIIIIYY + ((0.016786997725403114+0j))*IIIIIIIIIIXX + ((0.2987659097264251+0j))*IIIIIIIYYIII + ((0.2987659097264251+0j))*IIIIIIIXXIII + ((0.09572277709061083+0j))*IIIIIIIIIYYI + ((0.09572277709061083+0j))*IIIIIIIIIXXI + ((2.1400000000000006+0j))*IIIIIIIIIIII + ((-0.15350000000000008+0j))*ZIIIIIIIIIII + ((-0.15350000000000008+0j))*IIIIIIZIIIII + ((0.808+0j))*ZIIIIIZIIIII + ((-0.5717

In [13]:


E0_chain = np.linalg.eigvalsh(chain_mat_ham.to_matrix())[0]
print(f"star  E0 = {E0_direct:.10f}")
print(f"chain E0 = {E0_chain:.10f}")
assert np.isclose(E0_direct, E0_chain, atol=1e-9)

# Stronger check: the FULL spectra agree, not just the ground energy
spec_star = np.sort(np.linalg.eigvalsh(star_mat_ham.to_matrix()))
spec_chain = np.sort(np.linalg.eigvalsh(chain_mat_ham.to_matrix()))
assert np.allclose(spec_star, spec_chain, atol=1e-9)
print("Full 4096-eigenvalue spectra agree: the Lanczos-derived chain is exactly the same physics as the star.")

star  E0 = -5.6789498172
chain E0 = -5.6789498172


Full 4096-eigenvalue spectra agree: the Lanczos-derived chain is exactly the same physics as the star.


NameError: name 'chain_mat_ham' is not defined

## 6. Fixed-particle-number ground state

Reuses `ground_state(fixed_particle_number=...)` from `applications/` -- nothing AIM-specific
was needed for this to work.

In [14]:
n_elec = 6    
gs = ground_state(star_ham, fixed_particle_number=n_elec)
print(gs.expectation(star_ham.as_observable()).real) 
gs

-0.9582198606554312


0.2576|01110111⟩ + -0.5345|01111011⟩ + 0.0677|01111101⟩ + -0.0641|01111110⟩ + -0.5345|10110111⟩ + 0.5578|10111011⟩ + -0.0978|10111101⟩ + 0.0838|10111110⟩ + 0.0677|11010111⟩ + -0.0978|11011011⟩ + 0.0133|11011101⟩ + -0.0122|11011110⟩ + -0.0641|11100111⟩ + 0.0838|11101011⟩ + -0.0122|11101101⟩ + 0.0109|11101110⟩

## 9. A Majorana observable on the impurity (the Green's function's building block)

Eq. (12) of the source document writes the impurity Green's function in terms of correlators
of $\gamma_\sigma,\bar\gamma_\sigma$ -- the impurity's own Majorana pair. Computing the actual
time-dependent circuit (§3.1) is future work, but the observable itself is already exactly
`MajoranaTerm`/`jordan_wigner_majorana` from Task 20 -- nothing AIM-specific needed to define it.

In [15]:
# Majorana indices 0,1 are gamma_{d,up}, gamma-bar_{d,up} -- the impurity's up-spin Majorana pair
# (impurity_up is mode 0 under the canonical labeling, regardless of encoding).
impurity_majorana_obs = [majorana_bilinear(0, 1)]
encoded_obs = encoding.encode_majorana_terms(impurity_majorana_obs)

gs_small = ground_state(star_ham)
value = gs_small.expectation(encoded_obs)
print(f"<i*gamma_d,up*gamma-bar_d,up> in the ground state = {value:.6f}")
assert abs(value.imag) < 1e-10

<i*gamma_d,up*gamma-bar_d,up> in the ground state = 0.579053+0.000000j


## Summary

- `AndersonImpurity.star(...)` / `.chain(...)` are bare `FermionicSum`s (in fact `FermionicHamiltonian`s -- Hermiticity-asserted at construction), no qubits until `.encode(...)`, exactly like `FermiHubbard1D`.
- The two geometries are independently-specified baths, not interconvertible in `core/` -- but Lanczos-tridiagonalizing the star's bath energies (an ad-hoc, notebook-only computation, not library code) produces a chain that reproduces the star's *entire* spectrum exactly, for the real 5-bath material parameters, not just a toy case.
- Everything downstream of `.encode()` -- `ground_state()`/`trotter_circuit()` (free functions in `applications/`, not `Hamiltonian` methods -- `ground_state` because it needs a backend `Statevector`, `trotter_circuit` for symmetry with it), `fixed_particle_number`, Majorana observables -- is inherited unchanged; nothing AIM-specific was needed for any of it.
- `bath_interleaved_mode_order` gives an alternative qubit layout (baths first, impurity last) with no change in physics -- Step B, independent of Step A's mode-labeling convention.